<div align="center">

<a href="https://colab.research.google.com/github/utkukose/ai-applications-in-engineering-NB-lecture/blob/main/weeks/week-03/NB03_rule_based_systems.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

# Week 03: Knowledge, Rules and Expert Systems

**Artificial Intelligence Applications in Engineering (MUH-920 Mühendislikte Yapay Zeka Uygulamaları)**<br>Prof. Dr. Utku Kose<br>Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye<br>Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University<br>Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)<br>[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)<br>[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

<sub>Content licensed under CC BY 4.0, code under MIT. This course is updated in line with current developments in the field. Last update: October 2026.</sub>

</div>

## About this notebook

The notebook writes a forward-chaining and a backward-chaining engine in a few dozen lines, adds certainty factors and an explanation facility, and then encodes the Unified Soil Classification System for coarse and fine soils [1, 2]. A table of laboratory results is classified automatically and plotted on the plasticity chart, and the exercises extend the rule base.

This notebook is the hands-on part of Week 3. It opens with Python step 3: Dictionaries and functions and continues with the sections listed in the table of contents. The concepts are explained on the [lecture page](https://utkukose.github.io/ai-applications-in-engineering-NB-lecture/weeks/week-03/lecture.html) and in the [PDF notes](https://github.com/utkukose/ai-applications-in-engineering-NB-lecture/blob/main/weeks/week-03/Week03_Lecture_Notes.pdf). The [interactive lab](https://utkukose.github.io/ai-applications-in-engineering-NB-lecture/weeks/week-03/lab.html) holds the simulation, the self-assessment and the reflection, and the [week overview](https://github.com/utkukose/ai-applications-in-engineering-NB-lecture/blob/main/weeks/week-03/README.md) gives the learning outcomes, the study path, the discipline challenges and the tasks.

## How to use this notebook

Run the cells in order with Shift+Enter. No earlier Python experience is assumed: Python step 3 teaches the Python of the week first, and every further piece of Python appears in a cell marked **Python move**, which explains the syntax right where it is first needed. Exercise cells start with `None` placeholders and print feedback through `check(...)`. Reference solutions sit in collapsed cells: Open them only after a genuine attempt. Cells that download public data need an internet connection. When a source is unreachable, the notebook falls back to a documented synthetic stand-in so that every later cell still runs.

## 0. Setup

The last lines of the setup cell add a hint to every `NameError`. This error appears when a cell uses a name that no cell has defined in the current session, for example because the cells above have not run.

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})


def check(name, value, expected, tol=1e-6):
    """Compare an exercise answer with the reference value and print feedback."""
    if value is None:
        print(f"[{name}] Not attempted yet. Replace None with your answer and run the cell again.")
        return
    try:
        ok = abs(float(value) - float(expected)) <= tol * max(1.0, abs(float(expected)))
    except (TypeError, ValueError):
        ok = value == expected
    print(f"[{name}] {'Correct.' if ok else 'Not yet. Check the logic and run again.'}")

# Hint for a NameError: The usual cause in a notebook is a cell above that has not run in this session.
from IPython import get_ipython


def name_error_hint(shell, etype, value, tb, tb_offset=None):
    """Show a NameError as usual and add the hint."""
    shell.showtraceback((etype, value, tb), tb_offset=tb_offset)
    print("Hint: A name defined in a cell above exists only after that cell has run in this session. "
          "Runtime > Run before (Ctrl+F8) runs all cells above the current one.")


if get_ipython() is not None:
    get_ipython().set_custom_exc((NameError,), name_error_hint)

## Python step 3: Dictionaries and functions

> **Starting here?** Run the setup cell above first. The first code cell below checks this and stops with a message if the setup has not run in this session.

This step teaches the Python of the week with the week's own example. Read each explanation, run the cell below it and compare the output with the text; then change a value and predict the new result before running the cell again. A quick check and short exercises with immediate feedback close the step. The [Python path](https://github.com/utkukose/ai-applications-in-engineering-NB-lecture/blob/main/PYTHON_PATH.md) shows how the fourteen steps of the course build on each other.

### Dictionaries

A dictionary maps keys to values, like a small table with named entries. It is written with braces and colons, and values are read and written with square brackets and the key. The method `get` returns a default instead of an error when a key is missing, and `items` yields pairs of key and value for a loop. A laboratory result for a soil sample fits naturally into a dictionary.

In [ ]:
#@title Check that the setup cell has run
if "check" not in globals():
    raise RuntimeError("The setup cell above has not run in this session. "
                       "Run it first, then continue with this step.")

In [ ]:
sample = {"id": "BH2-3.5m", "fines_pct": 62.0, "LL": 48.0, "PL": 22.0}
print(sample["LL"], sample.get("organic", False))
sample["PI"] = sample["LL"] - sample["PL"]      # adds a new key
for key, value in sample.items():
    print(key, "=", value)

Dictionaries can hold other containers, for example a list of samples, or a dictionary of samples keyed by borehole and depth. They are also the everyday format of records read from files and web services.

### Functions

A function packages a calculation under a name so that it can be reused and tested. The keyword `def` starts the definition, the names in parentheses are parameters, and `return` sends the result back to the caller. The string on the first line of the body, the docstring, documents what the function does. Variables created inside a function are local and disappear when it returns. The A-line of the plasticity chart and a simplified group decision for fine-grained soils make two short functions [1].

In [ ]:
def a_line(ll):
    """Plasticity index on the A-line of the plasticity chart for liquid limit ll."""
    return 0.73 * (ll - 20)


def fine_soil_group(ll, pi):
    """Group symbol of an inorganic fine-grained soil (simplified)."""
    clay = pi > a_line(ll) and pi > 7
    if ll < 50:
        return "CL" if clay else "ML"
    return "CH" if clay else "MH"


print(round(a_line(48), 2))
print(fine_soil_group(48, 26), fine_soil_group(ll=62, pi=18))

The second call names its arguments, which makes it easier to read and independent of the order of the parameters. The full standard adds further cases, such as a dual symbol for low plasticity near the A-line and separate groups for organic soils; the simplified function shows the pattern, not the complete procedure. Parameters can also have default values, which a caller may omit.

In [ ]:
def plasticity_check(pl, ll, margin=0.0):
    """True when the liquid limit exceeds the plastic limit by more than the margin."""
    return ll - pl > margin


print(plasticity_check(22, 48), plasticity_check(22, 48, margin=30))

### Rules as data

Expert systems keep their knowledge as data, separate from the code that reasons with it. A list of dictionaries represents a small rule base for pump diagnosis, and a set holds the observed facts. The built-in function `all` returns `True` when every item of a sequence is true, so `all(c in facts for c in rule["if"])` asks whether every condition of a rule is among the facts. Repeating the pass until no rule adds a new fact is forward chaining, the inference method of this week.

In [ ]:
rules = [
    {"name": "R1", "if": ["low suction pressure", "crackling noise"], "then": "cavitation"},
    {"name": "R2", "if": ["high vibration", "peak at running speed"], "then": "imbalance"},
    {"name": "R3", "if": ["cavitation"], "then": "check suction line and NPSH margin"},
]
facts = {"low suction pressure", "crackling noise", "high vibration"}

changed = True
while changed:
    changed = False
    for rule in rules:
        if all(c in facts for c in rule["if"]) and rule["then"] not in facts:
            facts.add(rule["then"])
            print(rule["name"], "fired:", rule["then"])
            changed = True

Rule R2 does not fire, because one of its conditions is missing. Adding knowledge requires no change to the reasoning code, only a new dictionary in the list, which is the main practical advantage of keeping rules as data.

**Quick check.** After def area(b, h=0.5): return b * h, what is the value of area(2) + area(2, h=1)?

A. 3.0  
B. 2.0  
C. 4.0  
D. An error, because h is missing in the first call

<details><summary>Show the answer</summary>

**A.** The first call uses the default h = 0.5 and returns 1.0, and the second passes h = 1 and returns 2.

</details>

### Exercises for Python step 3

**Exercise 3.1.** Write a function `plasticity_index(ll, pl)` that returns the liquid limit minus the plastic limit, call it for LL = 41 and PL = 19, and store the result in `py_pi`.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _py3_pi_reference():
    return 41 - 19

In [ ]:
py_pi = None
check("Exercise 3.1", py_pi, _py3_pi_reference())

**Exercise 3.2.** Use `fine_soil_group` to classify a soil with LL = 35 and PI = 9, and store the group symbol in `py_group`. Then check the result by hand with the A-line.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _py3_group_reference():
    return "CL" if (9 > 0.73 * (35 - 20) and 9 > 7) else "ML"

In [ ]:
py_group = None
check("Exercise 3.2", py_group, _py3_group_reference())

**Exercise 3.3.** Add a rule R4 that concludes "balance the impeller" from "imbalance", add the fact "peak at running speed", and run the forward chaining loop again. Store the final number of facts in `py_n_facts`.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _py3_n_facts_reference():
    rs = [(["low suction pressure", "crackling noise"], "cavitation"), (["high vibration", "peak at running speed"], "imbalance"),
          (["cavitation"], "check suction line and NPSH margin"), (["imbalance"], "balance the impeller")]
    fs = {"low suction pressure", "crackling noise", "high vibration", "peak at running speed"}
    changed = True
    while changed:
        changed = False
        for cond, then in rs:
            if all(c in fs for c in cond) and then not in fs:
                fs.add(then)
                changed = True
    return len(fs)

In [ ]:
py_n_facts = None
check("Exercise 3.3", py_n_facts, _py3_n_facts_reference())

## 1. A knowledge base as a list of dictionaries

**Python move: lists of dictionaries.** Each rule is a dictionary with a name, a list of conditions, a conclusion and a certainty factor. The whole knowledge base is a list, so it can be printed, filtered and extended like any other data.

In [ ]:
RULES = [
    {"name": "R1", "if": ["high_vibration", "peak_1x"], "then": "unbalance", "cf": 0.7},
    {"name": "R2", "if": ["high_vibration", "peak_2x"], "then": "misalignment", "cf": 0.7},
    {"name": "R3", "if": ["crackling_noise", "low_suction_pressure"], "then": "cavitation", "cf": 0.8},
    {"name": "R4", "if": ["hot_bearing", "high_vibration"], "then": "bearing_wear", "cf": 0.6},
    {"name": "R5", "if": ["seal_leak"], "then": "seal_failure", "cf": 0.9},
    {"name": "R6", "if": ["cavitation"], "then": "action_raise_suction_head", "cf": 1.0},
    {"name": "R7", "if": ["misalignment"], "then": "action_realign", "cf": 1.0},
    {"name": "R8", "if": ["bearing_wear"], "then": "action_replace_bearing", "cf": 1.0},
]
print(len(RULES), "rules; conclusions:", sorted({r["then"] for r in RULES}))

## 2. Forward chaining with a trace

> **Starting here?** This section uses results of the cells above it, so choose **Runtime > Run before** (Ctrl+F8) first. The first code cell below checks this and stops with a message if they have not run in this session.

**Python move: sets and while loops.** The facts are a set. The loop runs while at least one rule adds a new fact, and `issubset` checks whether all conditions are known.

In [ ]:
#@title Check that the cells above have run
if not all(name in globals() for name in ["check", "RULES"]):
    raise RuntimeError("This section uses results of the cells above, which have not run in this session. "
                       "Choose Runtime > Run before (Ctrl+F8), then continue with this section.")

In [ ]:
def forward_chain(facts, rules, verbose=True):
    facts, trace, changed = set(facts), [], True
    while changed:
        changed = False
        for rule in rules:
            if set(rule["if"]).issubset(facts) and rule["then"] not in facts:
                facts.add(rule["then"])
                trace.append(rule["name"])
                changed = True
                if verbose:
                    print(f"{rule['name']}: IF {' AND '.join(rule['if'])} THEN {rule['then']}")
    return facts, trace


derived, trace = forward_chain({"high_vibration", "peak_2x", "hot_bearing"}, RULES)
print("\nNew facts:", sorted(derived - {"high_vibration", "peak_2x", "hot_bearing"}))

## 3. Backward chaining with recursion

> **Starting here?** This section uses results of the cells above it, so choose **Runtime > Run before** (Ctrl+F8) first. The first code cell below checks this and stops with a message if they have not run in this session.

**Python move: recursion.** A function may call itself. `prove(goal)` succeeds if the goal is a known fact, or if some rule concludes the goal and all of its conditions can be proved in turn.

In [ ]:
#@title Check that the cells above have run
if not all(name in globals() for name in ["check", "RULES"]):
    raise RuntimeError("This section uses results of the cells above, which have not run in this session. "
                       "Choose Runtime > Run before (Ctrl+F8), then continue with this section.")

In [ ]:
def prove(goal, facts, rules, depth=0):
    pad = "  " * depth
    if goal in facts:
        print(f"{pad}{goal}: known fact")
        return True
    for rule in rules:
        if rule["then"] == goal:
            print(f"{pad}{goal}: trying {rule['name']}")
            if all(prove(c, facts, rules, depth + 1) for c in rule["if"]):
                return True
    print(f"{pad}{goal}: cannot be established")
    return False


prove("action_realign", {"high_vibration", "peak_2x"}, RULES)

## 4. Certainty factors

> **Starting here?** This section uses results of the cells above it, so choose **Runtime > Run before** (Ctrl+F8) first. The first code cell below checks this and stops with a message if they have not run in this session.

A conclusion receives the rule's certainty times the smallest certainty of its conditions, and several supports combine as `CF1 + CF2 * (1 - CF1)` [3].

In [ ]:
#@title Check that the cells above have run
if not all(name in globals() for name in ["check", "RULES"]):
    raise RuntimeError("This section uses results of the cells above, which have not run in this session. "
                       "Choose Runtime > Run before (Ctrl+F8), then continue with this section.")

In [ ]:
def infer_cf(evidence, rules):
    cf = dict(evidence)
    for _ in range(len(rules)):                       # repeat so that chained rules propagate
        new = dict(evidence)
        for rule in rules:
            if all(c in cf for c in rule["if"]):
                support = rule["cf"] * min(cf[c] for c in rule["if"])
                old = new.get(rule["then"])
                new[rule["then"]] = support if old is None else old + support * (1 - old)
        cf = new
    return {k: round(v, 3) for k, v in cf.items() if k not in evidence}


print(infer_cf({"high_vibration": 0.9, "peak_2x": 0.8, "hot_bearing": 0.6}, RULES))

### Your turn, exercise 1

Add a rule `R9` that concludes `misalignment` with certainty 0.4 from `hot_bearing` and `peak_2x`. Recompute the certainty of `misalignment` for the evidence above and store it in `cf_misalignment`.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _cf_misalignment_reference():
    rules = RULES + [{"name": "R9", "if": ["hot_bearing", "peak_2x"], "then": "misalignment", "cf": 0.4}]
    return infer_cf({"high_vibration": 0.9, "peak_2x": 0.8, "hot_bearing": 0.6}, rules)["misalignment"]

In [ ]:
cf_misalignment = None   # your computation
check("Exercise 1", cf_misalignment, _cf_misalignment_reference(), tol=1e-3)

## 5. The Unified Soil Classification System as rules

The function follows ASTM D2487 for inorganic soils [1]: fine or coarse from the fines content, gravel or sand from the coarse fraction, gradation from Cu and Cc, and plasticity from the position relative to the A-line, PI = 0.73 (LL - 20) [2]. It returns the group symbol and the list of reasons.

**Python move: several return values.** `return symbol, reasons` returns a tuple, which the caller unpacks as `sym, why = uscs(...)`.

In [ ]:
def fine_group(ll, pi):
    a_line = 0.73 * (ll - 20)
    if ll < 50:
        if pi > 7 and pi >= a_line:
            return "CL"
        if 4 <= pi <= 7 and pi >= a_line:
            return "CL-ML"
        return "ML"
    return "CH" if pi >= a_line else "MH"


def uscs(fines, gravel_of_coarse, cu, cc, ll, pi):
    reasons = []
    if fines >= 50:
        reasons.append(f"{fines}% fines: fine grained")
        group = fine_group(ll, pi)
        reasons.append(f"LL {ll}, PI {pi}, A-line {0.73 * (ll - 20):.1f}: {group}")
        return group, reasons
    letter = "G" if gravel_of_coarse > 50 else "S"
    reasons.append(f"{fines}% fines: coarse grained, {'gravel' if letter == 'G' else 'sand'}")
    well = cu >= (4 if letter == "G" else 6) and 1 <= cc <= 3
    grading = letter + ("W" if well else "P")
    fg = fine_group(ll, pi)
    plastic = "C" if fg in ("CL", "CH") else ("CM" if fg == "CL-ML" else "M")
    if fines < 5:
        reasons.append(f"clean, Cu {cu}, Cc {cc}: {grading}")
        return grading, reasons
    if fines > 12:
        symbol = f"{letter}C-{letter}M" if plastic == "CM" else letter + plastic
        reasons.append(f"fines classify as {fg}: {symbol}")
        return symbol, reasons
    symbol = grading + "-" + letter + ("C" if plastic == "C" else "M")
    reasons.append(f"5 to 12% fines: dual symbol {symbol}")
    return symbol, reasons


sym, why = uscs(fines=8, gravel_of_coarse=60, cu=6.0, cc=1.5, ll=38, pi=18)
print(sym, why)

## 6. Classifying a table of laboratory results

> **Starting here?** This section uses results of the cells above it, so choose **Runtime > Run before** (Ctrl+F8) first. The first code cell below checks this and stops with a message if they have not run in this session.

**Python move: comprehensions.** A list comprehension applies the classifier to every sample in one line, and a dictionary comprehension counts the groups. The samples are illustrative values chosen to cover the chart.

In [ ]:
#@title Check that the cells above have run
if not all(name in globals() for name in ["check", "uscs"]):
    raise RuntimeError("This section uses results of the cells above, which have not run in this session. "
                       "Choose Runtime > Run before (Ctrl+F8), then continue with this section.")

In [ ]:
samples = [
    dict(id="BH1-2.0", fines=62, gravel_of_coarse=10, cu=0, cc=0, ll=41, pi=22),
    dict(id="BH1-4.5", fines=71, gravel_of_coarse=5, cu=0, cc=0, ll=58, pi=35),
    dict(id="BH2-1.0", fines=55, gravel_of_coarse=15, cu=0, cc=0, ll=32, pi=6),
    dict(id="BH2-3.0", fines=3, gravel_of_coarse=65, cu=8.1, cc=1.6, ll=0, pi=0),
    dict(id="BH3-1.5", fines=18, gravel_of_coarse=30, cu=0, cc=0, ll=29, pi=4),
    dict(id="BH3-6.0", fines=9, gravel_of_coarse=40, cu=4.2, cc=0.8, ll=45, pi=21),
    dict(id="BH4-2.5", fines=80, gravel_of_coarse=2, cu=0, cc=0, ll=66, pi=19),
    dict(id="BH4-5.0", fines=24, gravel_of_coarse=58, cu=0, cc=0, ll=36, pi=17),
]
groups = [uscs(s["fines"], s["gravel_of_coarse"], s["cu"], s["cc"], s["ll"], s["pi"])[0] for s in samples]
for s, g in zip(samples, groups):
    print(f"{s['id']:8s} -> {g}")
counts = {g: groups.count(g) for g in set(groups)}
print(counts)

In [ ]:
ll = list(range(10, 101))
plt.figure(figsize=(6, 4))
plt.plot(ll, [max(0, 0.73 * (x - 20)) for x in ll], label="A-line")
plt.plot(ll, [max(0, 0.9 * (x - 8)) for x in ll], ls="--", label="U-line")
plt.axvline(50, color="gray", lw=0.8, ls=":")
for s, g in zip(samples, groups):
    if s["fines"] >= 50 or s["fines"] > 12:
        plt.scatter(s["ll"], s["pi"], zorder=3)
        plt.annotate(f"{s['id']} {g}", (s["ll"], s["pi"]), fontsize=7, xytext=(3, 3), textcoords="offset points")
plt.xlabel("liquid limit LL"); plt.ylabel("plasticity index PI"); plt.ylim(0, 60); plt.legend(); plt.title("Samples on the plasticity chart"); plt.show()

### Your turn, exercise 2

How many of the eight samples are fine-grained soils that plot above the A-line? Use a comprehension with the condition `fines >= 50 and pi >= 0.73 * (ll - 20)` and store the count in `n_clays`.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _n_clays_reference():
    return sum(1 for s in samples if s["fines"] >= 50 and s["pi"] >= 0.73 * (s["ll"] - 20))

In [ ]:
n_clays = None   # your computation
check("Exercise 2", n_clays, _n_clays_reference())

### Your turn, exercise 3

Crisp thresholds make the classification jump. Find the smallest whole-number plasticity index at which a soil with 60 percent fines and LL = 40 changes from ML to CL. Store it in `pi_switch`, and explain in one sentence why a fuzzy boundary would describe borderline soils better.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _pi_switch_reference():
    for p in range(0, 61):
        if uscs(60, 0, 0, 0, 40, p)[0] == "CL":
            return p

In [ ]:
pi_switch = None   # your computation
check("Exercise 3", pi_switch, _pi_switch_reference())

## Next steps

Take the [self-assessment](https://utkukose.github.io/ai-applications-in-engineering-NB-lecture/weeks/week-03/lab.html#check) in the interactive lab, then answer the [reflection prompts](https://utkukose.github.io/ai-applications-in-engineering-NB-lecture/weeks/week-03/lab.html#reflect) and export the learning log. The [discipline challenges](https://github.com/utkukose/ai-applications-in-engineering-NB-lecture/blob/main/weeks/week-03/README.md#discipline-challenges), the [weekly task](https://github.com/utkukose/ai-applications-in-engineering-NB-lecture/blob/main/weeks/week-03/README.md#weekly-task) and the [research assignment](https://github.com/utkukose/ai-applications-in-engineering-NB-lecture/blob/main/weeks/week-03/README.md#research-and-report-assignment-optional) are listed in the week overview, and [Week 4](https://github.com/utkukose/ai-applications-in-engineering-NB-lecture/blob/main/weeks/week-04/README.md) starts with its overview.

## References

[1] ASTM International (2017). *ASTM D2487-17: Standard Practice for Classification of Soils for Engineering Purposes (Unified Soil Classification System)*. ASTM International.

[2] Casagrande, A. (1948). Classification and identification of soils. *Transactions of the American Society of Civil Engineers*, *113*, 901-930.

[3] Buchanan, B. G., & Shortliffe, E. H. (Eds.) (1984). *Rule-Based Expert Systems: The MYCIN Experiments of the Stanford Heuristic Programming Project*. Addison-Wesley.